# A1: Decision-model outputs

**Question (RQ1A).** How do the 17 decision models behave on identical requests, before any tabular predictor is involved? How confident are their answers, how typical is each model relative to the others, and how good are its probabilities on their own?

Binary requests give one probability, multiclass requests one per class, regression requests nine probabilities over ordered evidence levels (−4 … +4). Every table comes from **one compact export**, `output_JEVPFN/A/exp01_outputs/results/outputs/comparison.zip` (schema `jevpfn.rq1a.report.v2`), built on VSC by the Mindwell CPU job `scripts/slurm/rq1_outputs.slurm` from the complete feature stores. This notebook reads only that ZIP and recomputes nothing; a missing or superseded-schema ZIP gives a one-line waiting state per section.

**Stand-alone quality and regression (decision of 23 November 2026).** The export scores the stored *joint*-request probabilities against the target without fitting or recalibration. Log loss, Brier and ECE exist only for the nine classification datasets; for the nine regression datasets the export has only the Spearman correlation between the expected level and the target (log loss is undefined and no ranked probability score is exported for A1). A1 therefore never pools the two families: classification quality has N = 9 (ROC-AUC in the figures, log loss in the descriptive CD diagram), regression is a separate rank-only ρ with N = 9. The pooled proper-score skill belongs to A2. Labels are used only for this description and never select a model, feature or predictor.

In [1]:
import sys
from pathlib import Path

REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "src").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import pandas as pd

from src.evaluation import comparison_report as report
from src.evaluation import group_a, learner_scores, model_properties, zero_shot
from src.utils.config import save_resolved_config
from src.utils.notebook_display import display_json, display_table
from src.utils.notebook_report import begin_report, finish_report, record, report_section
from src.visualize import critical_difference, figures, style
from src.visualize import outputs as plots
from src.visualize.group_b import render
from src.visualize.zero_shot import direct_score_figures

style.apply()
NOTEBOOK = "A/exp01_outputs"
begin_report()
save = figures.FigureSaver(NOTEBOOK)
cfg = report.report_config()
save_resolved_config(cfg, NOTEBOOK)
report_section("1. What is complete")
result, receipt = report.read_report(cfg, allow_previous_roster=True)
available = receipt["status"].startswith("verified")
print(report.describe_receipt(receipt))
if receipt.get("unassessed_additions"):
    print("Models awaiting a new export:", ", ".join(receipt["unassessed_additions"]))
record({k: receipt.get(k) for k in ("status", "path", "created_utc", "origin", "sha256", "bytes", "schema", "receipt_sha256")}, "Export identity")
if available:
    inventory = result.table("inventory")
    blocks = (inventory.assign(available=inventory.status.eq("available")).groupby("provider", sort=False)["available"]
              .agg(["sum", "size"]).rename(columns={"sum": "Complete blocks", "size": "Expected blocks"}))
    blocks.index = [style.DECISION_SHORT.get(p, p) for p in blocks.index]
    display_table("Complete feature blocks per decision model (dataset x request mode)", blocks.rename_axis("Decision model").reset_index(), preview_rows=17)
    record(inventory, "Exact immutable feature artifact identities")
shape = result.table("output_shape") if available else pd.DataFrame()
loo = result.table("loo_consensus") if available else pd.DataFrame()
quality = result.table("quality") if available else pd.DataFrame()

comparison.zip: created 2026-11-18T13:24:39.664228+00:00 by verified local export of 11 complete VSC models, 2026-11-18, 1,912,238 bytes, SHA-256 89c4986f83ba69546646a33b09ee4aa4ed8d94f9bd95f2d17050822d288cb8dd. Schema jevpfn.rq1a.report.v1 predates jevpfn.rq1a.report.v2; waiting for a new export. No measurements are shown.


## 2. Who the models are

Size, readout type and architecture family come from the sourced properties file (`config/02_feature_creation/model_properties.yaml`); a value no local source states stays missing. Compute time per 1,000 decision requests (successful decision-feature GPU-hours in the Slurm ledger divided by the distinct requests of the 0A token audit) arrives with the export. This table orders every Group A figure: rows are grouped by architecture family, then by size.

In [2]:
report_section("2. Who the models are")
profile = result.table("model_profile") if available else pd.DataFrame()
if profile.empty:
    profile = model_properties.profile_table()
    print("Compute time is waiting for the export; properties are read from the configuration.")
properties = model_properties.profile_table()
order = group_a.model_order(properties.provider, properties)
shown = profile.set_index("provider").reindex(order).reset_index()
shown.insert(0, "model", shown.provider.map(lambda p: style.DECISION_SHORT.get(p, p)))
display_table("Model profile: size, readout, architecture and GPU-hours per 1,000 requests (missing = not recorded)",
              shown.drop(columns=["provider"]).round(4), preview_rows=17)
record(shown, "Model profile")

Compute time is waiting for the export; properties are read from the configuration.


model,display_name,parameters_billion,readout_type,architecture_family,gpu_hours_per_1000_requests,compute_status
Kev 27B,kev-27b,25.600,pointer_head,decoder_pointer_head,NaN,waiting for export
Clef-flash,clef-flash,9.410,pointer_head,decoder_pointer_head,NaN,waiting for export
Strands 2B,strands-decider-2B-qwen3.5-v1-2610,NaN,pointer_head,decoder_pointer_head,NaN,waiting for export
Liquid d1-3B,d1-3B,3.120,lm_label_tokens,decoder_lm_head,NaN,waiting for export
Intern 4B,Intern-Decision-4B,NaN,lm_label_tokens,decoder_lm_head,NaN,waiting for export
SII 27B,SII-Decision-27B-Full,NaN,lm_label_tokens,decoder_lm_head,NaN,waiting for export
Nimble 9B,Bespoke-Nimble-9B,NaN,lm_label_tokens,decoder_lm_head,NaN,waiting for export
Sieve 31B v3,Sieve-31B-v3,31.000,None,bidirectional_encoder,NaN,waiting for export
Mammoth 27B,mammoth-1,27.000,lm_label_tokens,bidirectional_encoder,NaN,waiting for export
Drex Max 27B,drex-max-v1,27.000,None,bidirectional_encoder,NaN,waiting for export


## 3. Summary: one profile per model

One row per decision model and five views of its joint-request outputs: (a) confidence, (b) normalized entropy, (c) typicality, the share of inputs on which the model gives the same top answer as the leave-one-out consensus of all other models, and its stand-alone quality on (d) classification and (e) regression datasets. Grey dots are datasets, so the spread across datasets stays visible next to the median. The consensus is not the truth; related families (three Qwen3.8-27B derivatives) can dominate it.

In [3]:
report_section("3. Summary: model profile")
values = group_a.a1_dataset_values(shape, loo, quality)
if values.empty:
    print("Waiting for export: the model profile needs the output-shape, consensus and quality tables of comparison.zip.")
else:
    render(save, plots.model_profile_strip(values, properties, order=order))
    medians = values.groupby(["provider", "metric"]).value.median().unstack("metric").reindex(order)
    medians.index = [style.DECISION_SHORT.get(p, p) for p in medians.index]
    display_table("Median over datasets per model (joint requests)", medians.round(3).reset_index(names="model"), preview_rows=17)
    record(medians.reset_index(names="model"), "A1 profile medians per model")

Waiting for export: the model profile needs the output-shape, consensus and quality tables of comparison.zip.

## 4. Details

**Output shape per dataset** (joint requests): the same concentration measures with one row per dataset, and, for regression, how much of the nine-level scale each model uses. **Pairwise agreement** (joint requests): share of identical top answers and the distance between probability vectors (total variation; earth mover's distance over the ordered levels for regression), with models ordered by hierarchical clustering so similar models sit together. **Stand-alone quality and calibration**: per dataset, with reliability diagrams summarised by the median and interquartile band over models; Strands 2B, the RQ3 candidate, is drawn individually.

In [4]:
report_section("4. Details")
if shape.empty:
    print("Waiting for export: no output-shape table is available.")
else:
    render(save, plots.output_shape_figure(shape, "joint"))
    render(save, plots.regression_levels_figure(shape, "joint"))
    table = report.dataset_mode_table(shape, report.OUTPUT_SHAPE_METRICS)
    record(table, "Output shape per model, dataset and request mode")
    record(result.table("level_histogram"), "Distribution of the expected regression level (bins of 0.5 levels)")
pairs = result.table("pairs") if available else pd.DataFrame()
if pairs.empty or "top_agreement" not in pairs:
    print("Waiting for export: no pairwise top-answer agreement table is available.")
else:
    render(save, plots.pairwise_figures(pairs, "joint"))
    record(report.pairwise_table(pairs), "Pairwise agreement per task and mode, equal dataset weight")
if quality.empty:
    print("Waiting for export: no stand-alone quality table is available.")
else:
    render(save, direct_score_figures(quality, result.table("reliability"), highlight=("strands",)))
    summary = zero_shot.summary_table(quality)
    display_table("Stand-alone scores: median and range over decision models (reference only)", summary.round(3), preview_rows=12)
    record(zero_shot.REFERENCE_LABEL + ". Labels are used only for scoring.", "Evidence boundary")
    record(summary, "Stand-alone scores per dataset (median and range over models)")
    record(quality.drop(columns="label"), "Every stand-alone score, coverage and pending artifact")
if not loo.empty:
    record(report.equal_dataset_table(loo, report.CONSENSUS_METRICS), "Leave-one-out consensus per task and mode, equal dataset weight")

Waiting for export: no output-shape table is available.
Waiting for export: no pairwise top-answer agreement table is available.
Waiting for export: no stand-alone quality table is available.


## 5. Appendix: per-column requests

The same output-shape view on per-column requests (one text field per request; fields averaged with equal weight within a dataset).

In [5]:
report_section("5. Appendix: per-column requests")
if shape.empty:
    print("Waiting for export: no output-shape table is available.")
else:
    render(save, plots.output_shape_figure(shape, "per_column"))

Waiting for export: no output-shape table is available.


## 6. Statistics

Framework application `A1_outputs` from `config/05_synthesis/analysis.yaml`. The owner approved the framework on 24-11-2026; while the export is absent this section prints its waiting state. It ranks the 17 models by stand-alone log loss on the nine classification datasets (descriptive Friedman ranks and critical-difference diagram); regression has no log loss and is not ranked here.

In [6]:
report_section("6. Statistics")
framework = learner_scores.run_application("A1_outputs")
print(f"A1_outputs: {framework['status']} (framework.status = {framework['framework_status']})")
display_json("Framework application A1_outputs", learner_scores.summary_record(framework))
render(save, critical_difference.cd_figures(framework))

A1_outputs: awaiting_export (framework.status = approved)


## Complete report

In [7]:
finish_report(NOTEBOOK, save.summary() + "\nOnly the export ZIP and the model-properties configuration were read; no inference, model training or VSC changes were performed.")

A/exp01_outputs

1. What is complete


Export identity
{
  "status": "superseded_schema",
  "path": "C:\\Users\\U0152019\\PhD Documents\\Projects\\5. JEVPFN\\JEVPFN\\output_JEVPFN\\A\\exp01_outputs\\results\\outputs\\comparison.zip",
  "created_utc": "2026-11-18T13:24:39.664228+00:00",
  "origin": "verified local export of 11 complete VSC models, 2026-11-18",
  "sha256": "89c4986f83ba69546646a33b09ee4aa4ed8d94f9bd95f2d17050822d288cb8dd",
  "bytes": 1912238,
  "schema": "jevpfn.rq1a.report.v1",
  "receipt_sha256": null
}


2. Who the models are


Model profile: size, readout, architecture and GPU-hours per 1,000 requests (missing = not recorded)
         model                       display_name  parameters_billion    readout_type   architecture_family  gpu_hours_per_1000_requests     compute_status
       Kev 27B                            kev-27b              25.600    pointer_head  decoder_pointer_head                          NaN waiting for export
    Clef-flash                     